# TXF Overnight → 0050 Open v0.1.2

Modes: `smoke` synthetic framework test; `pilot` small real-data run (≤14 calendar days); `validation` bounded real-data QA (≤100 calendar days); `full` explicitly confirmed formal research.

For Phase A run `validation` with 2026-09-01 to 2026-09-07, then Phase B 2026-07-01 to 2026-09-30. Validation intentionally produces no predictive statistical inference.


In [ ]:
MODE = 'smoke'  # smoke / pilot / validation / full
START = '2026-09-01'  # validation Phase B: '2026-07-01'
END = '2026-09-07'  # validation Phase B: '2026-09-30'
CONFIRM_FULL = False
REQUEST_SPACING_SECONDS = 1.1
FUBON_WHEEL = ''  # optional: verified local wheel override; default auto-download by Drive File ID
FOLDER_ID = '1E9VkK_NgePZUidKu6ZqB2gceLnKy9kYx'

In [ ]:
import os, sys, subprocess, pathlib, tempfile
from google.colab import drive, userdata, auth
drive.mount('/content/drive')
repo = pathlib.Path('/content/txf_overnight_spot_open')
# Token 經 askpass 傳遞，不放 URL、命令列或 git remote。
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = ''
with tempfile.TemporaryDirectory() as td:
    ask = pathlib.Path(td) / 'askpass.py'
    ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ.get("TXF_CLONE_TOKEN", ""))\n')
    ask.chmod(0o700)
    env = dict(os.environ, GIT_ASKPASS=str(ask), GIT_TERMINAL_PROMPT='0', TXF_CLONE_TOKEN=token or '')
    if repo.exists():
        clean = subprocess.check_output(['git','-C',str(repo),'status','--porcelain'],text=True)
        remote = subprocess.check_output(['git','-C',str(repo),'remote','get-url','origin'],text=True).strip()
        branch = subprocess.check_output(['git','-C',str(repo),'branch','--show-current'],text=True).strip()
        if clean or remote != 'https://github.com/hh4832/txf_overnight_spot_open.git' or branch != 'main':
            raise RuntimeError('Existing checkout differs or is dirty; preserve it and resolve first')
        cmd = ['git','-C',str(repo),'pull','--ff-only','origin','main']
    else:
        cmd = ['git','clone','--branch','main','https://github.com/hh4832/txf_overnight_spot_open.git',str(repo)]
    result = subprocess.run(cmd,env=env,capture_output=True)
    if result.returncode:
        raise RuntimeError('GitHub clone/update failed; verify repository access and Secret')
del token, env
os.chdir(repo)
# Colab retains imports after git pull; never silently run stale code.
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
if any(name in sys.modules for name in ('src.pipeline', 'src.config', 'src.build_event_dataset')):
    raise RuntimeError('Repository may have updated after import: restart Colab runtime and Run All')
if sys.version_info < (3,11): raise RuntimeError('Python >=3.11 required')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
print('Git SHA:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
# Authenticate Drive before attempting the private SDK download.
from google.colab import userdata, auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
service = build('drive', 'v3', credentials=credentials, cache_discovery=False)

if MODE != 'smoke':
    from src.sdk_install import ensure_fubon_sdk
    sdk_version = ensure_fubon_sdk(service=service, override=FUBON_WHEEL)
    print('Fubon SDK ready:', sdk_version)
    for name in ('FINLAB_API_TOKEN','FUBON_PERSONAL_ID','FUBON_API_KEY','FUBON_CERT_PATH','FUBON_CERT_PASSWORD'):
        try:
            value = userdata.get(name)
        except Exception:
            raise RuntimeError(name + ': unavailable') from None
        if not value:
            raise RuntimeError(name + ': missing')
        os.environ[name] = value
    del value

In [ ]:
from src.drive_io import validate_folder, verified_cache_root, upload_run
validate_folder(service,FOLDER_ID)
cache_root = verified_cache_root(service,FOLDER_ID,'/content/drive/MyDrive')
print('Target Folder ID verified:', FOLDER_ID)

In [ ]:
from src.config import load
from src.pipeline import run
if MODE == 'validation':
    assert 'validation' in pathlib.Path('src/config.py').read_text(), 'Stale validation mode'
cfg = load(mode=MODE,start=START,end=END,confirm_full=CONFIRM_FULL,
           request_spacing_seconds=REQUEST_SPACING_SECONDS)
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
# Research outputs are local staging, raw cache resides in the verified Drive folder.
output = pathlib.Path('/content/txf_research_outputs')
before = set((output/'runs').glob('*'))
try:
    run_dir = run(cfg,output,cache_root=cache_root)
except Exception:
    failed = set((output/'runs').glob('*')) - before
    for folder in failed:
        upload_run(service,FOLDER_ID,folder)
    raise
uploaded_id = upload_run(service,FOLDER_ID,run_dir)
print('Completed:', MODE, 'https://drive.google.com/drive/folders/' + uploaded_id)
from IPython.display import Markdown, display
display(Markdown((run_dir/'reports/summary.md').read_text()))
